# Baseline Network

Build, train & evaluate a simple CNN as our starting point. All later networks are compared against this baseline.

In [ ]:
# reload src/config.py automatically whenever it changes
%load_ext autoreload
%autoreload 2

# import project settings
import sys
sys.path.append("..")
from src import config

# import required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from src.plots import plot_training_history
from pathlib import Path
from sklearn.metrics import ConfusionMatrixDisplay
from PIL import Image

In [ ]:
# check TensorFlow version
print(tf.__version__)

In [ ]:
# make results repeatable
tf.keras.utils.set_random_seed(config.RANDOM_SEED)

## Data Pipeline

Load images from the training, validation & test folders.

In [ ]:
# training data - shuffled so each batch contains a random mix of classes
training_set = tf.keras.utils.image_dataset_from_directory(
    config.TRAINING_DIR,
    image_size = (config.IMG_HEIGHT, config.IMG_WIDTH),
    batch_size = config.BATCH_SIZE,
    label_mode = 'categorical',
    shuffle = True,
    seed = config.RANDOM_SEED
)
# validation data
validation_set = tf.keras.utils.image_dataset_from_directory(
    config.VALIDATION_DIR,
    image_size = (config.IMG_HEIGHT, config.IMG_WIDTH),
    batch_size = config.BATCH_SIZE,
    label_mode = 'categorical',
    shuffle = False
)
# test data
test_set = tf.keras.utils.image_dataset_from_directory(
    config.TEST_DIR,
    image_size = (config.IMG_HEIGHT, config.IMG_WIDTH),
    batch_size = config.BATCH_SIZE,
    label_mode = 'categorical',
    shuffle = False
)




In [ ]:
# class names
class_names = training_set.class_names
print(class_names)

In [ ]:
# check one batch
for images, labels in training_set.take(1):
    #print(images)
    print(f"Images shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"First label: {labels[0].numpy()}")



## Network Architecture

Our baseline network is simple, but gives us a starting point to refine from: 2 Convolutional Layers (32 filters each) with Max Pooling, a single Dense layer with 32 neurons, and our output layer.

In [ ]:
# network architecture (Functional API)
# input: image of 80 x 60 pixels, 3 color channels
inputs = tf.keras.Input(shape=(config.IMG_HEIGHT,config.IMG_WIDTH, config.NUM_CHANNELS))

# rescale pixle values to range of 0-1
x = tf.keras.layers.Rescaling(1./255) (inputs)


# the first convolutional block
# convolutional layer
x = tf.keras.layers.Conv2D(filters= 32, 
                           kernel_size= (3,3),
                           activation='relu',
                           padding='same',
                           name='Conv_1') (x)

# pooling layer
x = tf.keras.layers.MaxPooling2D(name='Pool_1')(x)
# end of first convolutional block

# the second convolutional block
# convolutional layer
x = tf.keras.layers.Conv2D(filters= 32, 
                           kernel_size= (3,3),
                           activation='relu',
                           padding='same',
                           name='Conv_2') (x)

# pooling layer
x = tf.keras.layers.MaxPooling2D(name='Pool_2')(x)
# end of second convolutional block

# flatten layer
x = tf.keras.layers.Flatten()(x)
    
# fully-connected (dense) ReLU layer
x = tf.keras.layers.Dense(32, activation="relu")(x)

# output softmax layer
outputs = tf.keras.layers.Dense(config.NUM_CLASSES, activation="softmax")(x)

# connect inputs to outputs to create the model
model = tf.keras.Model(inputs, outputs)

# view network architecture
model.summary()


In [ ]:
# compile the model
model.compile(loss='categorical_crossentropy',
             optimizer='adam',
             metrics=['accuracy'])

In [ ]:
# save the best model (highest validation accuracy) during training
model_filename = config.MODELS_DIR / "fashion_cnn_v01.keras"     

save_best_model = tf.keras.callbacks.ModelCheckpoint(
    filepath = model_filename,
    monitor = "val_accuracy",        
    mode = "max",           
    save_best_only = True,
    verbose = 0
)

In [ ]:
# train the network
history = model.fit(
    training_set,                   
    validation_data = validation_set,
    epochs = config.NUM_EPOCHS,
    callbacks = [save_best_model],
    shuffle= False # training_set is already shuffled
)

In [ ]:
plot_training_history(history,
                      title="Baseline Network",
                      save_path=config.FIGURES_DIR / "baseline-accuracy-plot.png")

In [ ]:
# load the best model saved during training (highest validation accuracy)
best_model = tf.keras.models.load_model(model_filename)

In [ ]:
# evaluate on the test set - images never used in training or in choosing the best epoch
test_loss, test_accuracy = best_model.evaluate(test_set, verbose=0)

print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test loss: {test_loss:.4f}")

In [ ]:
# predicted probabilities for every test image, one column per class
probabilities_df = pd.DataFrame(best_model.predict(test_set, verbose=0), columns=class_names)

# true labels for every test image (same order, because test_set has shuffle=False)
actual_df = pd.DataFrame(np.concatenate([labels for _, labels in test_set]), columns=class_names)

# create dataframe to analyse
predictions_df = pd.DataFrame({"actual_label": actual_df.idxmax(axis=1),
                               "predicted_label": probabilities_df.idxmax(axis=1),
                               "predicted_probability": probabilities_df.max(axis=1),
                               "filename": [Path(path).name for path in test_set.file_paths]})

predictions_df["correct"] = np.where(predictions_df["actual_label"] == predictions_df["predicted_label"], 1, 0)

In [ ]:
predictions_df.head()

In [ ]:
print(predictions_df['correct'].mean())

In [ ]:


# confusion matrix as a heatmap - rows = actual class, columns = predicted class
fig, ax = plt.subplots(figsize=(8, 7))

ConfusionMatrixDisplay.from_predictions(
    predictions_df["actual_label"],
    predictions_df["predicted_label"],
    labels=config.CLASSES,       
    cmap="Blues",
    colorbar=False,
    ax=ax
)

ax.set_title("Baseline Network - Test Set Confusion Matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "baseline-confusion-matrix.png", dpi=150)
plt.show()

In [ ]:
# the model's mistakes, most confident first
mistakes = predictions_df.loc[predictions_df["correct"] == 0].sort_values("predicted_probability", ascending=False)

print(f"Number of mistakes: {len(mistakes)}")
mistakes.head(10)

In [ ]:
# mistakes where the actual and predicted class are the two shoe classes
shoe_classes = ["Casual Shoes", "Sports Shoes"]

shoe_confusion = mistakes.loc[mistakes["actual_label"].isin(shoe_classes) & mistakes["predicted_label"].isin(shoe_classes)]

print(f"Shoe vs shoe mistakes: {len(shoe_confusion)} of {len(mistakes)} ({len(shoe_confusion) / len(mistakes):.0%})")

## Error Analysis

Overall accuracy hides *why* the network gets things wrong. Here we look at the network's most confident mistakes to see whether they are caused by the network, or by the data itself.

In [ ]:
# show the 8 most confident mistakes
fig, axes = plt.subplots(2, 4, figsize=(10, 7))

for ax, (_, row) in zip(axes.flat, mistakes.head(8).iterrows()):
    image_path = config.TEST_DIR / row["actual_label"] / row["filename"]
    ax.imshow(Image.open(image_path).convert("RGB"))
    ax.set_title(f"Actual: {row['actual_label']}\nPredicted: {row['predicted_label']}\n({row['predicted_probability']:.0%})", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "baseline-mistakes.png", dpi=150)
plt.show()

In [ ]:
# random sample of 12 Sports Shoes & 12 Casual Shoes from the test set - chosen at random, NOT from the mistakes
shoe_sample = (predictions_df.loc[predictions_df["actual_label"].isin(["Sports Shoes", "Casual Shoes"])]
               .groupby("actual_label")
               .sample(n=12, random_state=config.RANDOM_SEED))

fig, axes = plt.subplots(4, 6, figsize=(12, 9))

for ax, (_, row) in zip(axes.flat, shoe_sample.iterrows()):
    image_path = config.TEST_DIR / row["actual_label"] / row["filename"]
    ax.imshow(Image.open(image_path).convert("RGB"))
    ax.set_title(row["actual_label"], fontsize=9)
    ax.axis("off")

plt.suptitle("Random sample of shoe labels - could you tell them apart?")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "shoe-label-sample.png", dpi=150)
plt.show()

## Summary

**Architecture**

* 2 Convolutional blocks (32 filters, 3 x 3, ReLU) each followed by Max Pooling
* 1 Dense layer (32 neurons, ReLU) & a softmax output layer (6 classes)
* 317,574 parameters

**Training**

* Trained for 50 epochs, best model saved at epoch 42
* Best validation accuracy: **92.3%**
* Strong overfitting - training accuracy reached 100%, and validation loss rose steadily after epoch 7

**Test Set Performance**

* Overall accuracy: **92.3%** (554 / 600)

| Class | Accuracy |
|---|---|
| Handbags | 99% |
| Watches | 98% |
| Shirts | 95% |
| Tshirts | 94% |
| Casual Shoes | 89% |
| Sports Shoes | 79% |

* Confusion between Sports Shoes & Casual Shoes accounts for around two-thirds (31 of 46) of all mistakes
* The network is overconfident - its most confident mistakes were predicted with almost 100% certainty

**Error Analysis**

* Several mistakes are caused by the data, not the network - e.g. a Watch image that only shows clothing, and shoes that could reasonably belong to either class
* In a random sample of 24 test shoe images, 7 could reasonably belong to either shoe class
* Because some labels are unclear, there is a limit to how accurate any network can be here - especially for shoes! We kept every test image as it was, so our results stay honest

**Next:** `03_dropout.ipynb` - add Dropout to tackle overfitting